# This notebook generates KG ans CSV files from tabular data (Google Sheets)

In [ ]:
import gspread
import rdflib
from rdflib import Graph, Namespace, Literal, URIRef, BNode
from rdflib.namespace import RDF, XSD, PROV, DCTERMS
from rdflib.util import _iri2uri
import csv
import datetime as dt

### Import spreadsheet data

In [ ]:
def load_spreadsheet(service_account:str, sh_name:str) -> dict:
    '''
    Loading tabs from Google Sheets
    Returns a dict with the data from all tabs
    service_account: str, filename of the .json with credentials to Google Sheets
    sh_name: str, spreadsheet name
    '''
    
    sh_data = {}
    
    gc = gspread.service_account(filename=service_account)
    sh = gc.open(sh_name)

    tab_names_mapping = {"cases":"all_records",
                         "defendants":"defendants",
                         "penalties":"penalties",
                         "mapping_cases_sources":"cases_sources",
                         "sources":"sources",
                         "mapping_cases_mentioned_in":"cases_mentioned",
                         "mentioned_in":"mentioned",
                         "mapping_trigger_obj":"cases_trigger",
                         "trigger_obj":"triggers",
                         "regions":"regions",
                         "courts":"courts",
                         "articles":"articles",
                         "monitorings":"monitorings"}

    for tab_name, data_key in tab_names_mapping.items():
        sh_data[data_key] = sh.worksheet(tab_name).get_all_records()

    return sh_data

### Utility functions

In [ ]:
def list_to_str(input_list:list) -> str:

    out_str = ""

    if len(input_list) > 1:
        for i in input_list:
            out_str += f"{i}; "
        
    if len(input_list) == 1:
        out_str = str(input_list[0])

    return out_str.rstrip('; ')

In [ ]:
def get_legal_article(art_num:str) -> str:
    art_split = art_num.split('_')
    art_uri = f"article_{art_split[0].replace('.','_')}_part_{art_split[1]}"
    return art_uri

In [ ]:
def get_info_per_record(sh_data:dict, record_id:str, info_type:str) -> list:
    '''
    Getting case data associated with record_id
    !NB: requires the 'sh_data' dict
    info_type: str; one of these: "penalties", "cases_sources", "cases_mentioned", "cases_trigger"
    '''
    
    info_type_keys = {
        "penalties": "penalty_id",
        "cases_sources": "source_id",
        "cases_mentioned": "mentioned_in_id",
        "cases_trigger": "trigger_object_id"
    }
    info_type_key = info_type_keys[info_type]
    
    return [
        info[info_type_key]
        for info in sh_data[info_type]
        if info["record_id"] == record_id
    ]

In [ ]:
def save_kg(graph, path_to_save:str, filename:str) -> str:
    '''
    Saving a KG in four formats: Turtle, JSON-LD, XML/RDF, N-Triples
    '''
    
    graph.serialize(destination=f"{path_to_save}/{filename}.ttl", format="turtle")
    graph.serialize(destination=f"{path_to_save}/{filename}.jsonld", format="json-ld")
    graph.serialize(destination=f"{path_to_save}/{filename}.rdf", format="pretty-xml")
    graph.serialize(destination=f"{path_to_save}/{filename}.nt", format="nt11", encoding='utf-8') # UTF8 encoded

    return f"{filename} is saved in {path_to_save}"

## Functions: generating main KG files

In [ ]:
def generate_main(sh_data:dict, path_to_save:str) -> str:
    '''
    Generating the main KG in TTL, JSON LD, RDF/XML, N Triples
    Returns a status str
    '''

    # Namespace declarations
    DATAOUT = Namespace("https://dataout.org/ontology#")
    P621 = Namespace("https://dataout.org/rdf/data/project-6-21#")
    SCHEMA = Namespace("https://schema.org/")
    LAW = Namespace("https://dataout.org/rdf/reference/ru_law#")
    REGION = Namespace("https://dataout.org/rdf/reference/ru_regions#")
    COURT = Namespace("https://dataout.org/rdf/reference/ru_courts#")
    
    main = Graph()

    main.bind("dataout", DATAOUT)
    main.bind("p621", P621)
    main.bind("schema", SCHEMA)
    main.bind("law", LAW)
    main.bind("region", REGION)
    main.bind("court", COURT)

    # generating main triples
    
    for record in sh_data["all_records"]:
    
        subject = P621[record["record_id"]]
        
        main.add((subject, RDF.type, DATAOUT.MonitoringRecord))
        main.add((subject, PROV.wasGeneratedBy, P621[record['monitoring_id']]))
        main.add((subject, DATAOUT.monitoringRecordNumber, Literal(record["case_number"])))
        main.add((subject, SCHEMA.identifier, Literal(record["record_id"])))
        main.add((subject, DATAOUT.concernsLegalArticle, LAW[get_legal_article(record["article"])]))
        main.add((subject, DATAOUT.monitoredIncidentYear, Literal(record["year"], datatype=XSD.gYear)))
        main.add((subject, SCHEMA.addressRegion, REGION[str(record["region"])]))
        main.add((subject, DATAOUT.hasDefendant, P621[str(record["defendant_id"])]))
        main.add((subject, DATAOUT.hasSnippet, Literal(record["snippet_en"], lang="en")))
        # check RU snippet
        if record["snippet_ru"] != "":
            main.add((subject, DATAOUT.hasSnippet, Literal(record["snippet_ru"], lang="ru")))
        
        # getting penalties
        penalties_per_record = get_info_per_record(sh_data, record["record_id"], 'penalties')
        for pen in penalties_per_record:
            main.add((subject, DATAOUT.hasPenalty, P621[pen]))
        
        # getting trigger objects
        triggers_per_record = get_info_per_record(sh_data, record["record_id"], 'cases_trigger')
        for tr in triggers_per_record:
            main.add((subject, DATAOUT.hasTriggerObject, P621[str(tr)]))
        
        # getting sources
        sources_per_record = get_info_per_record(sh_data, record["record_id"], 'cases_sources')
        for source in sources_per_record:
            main.add((subject, PROV.wasDerivedFrom, P621[source]))
        
        # getting mentioned_in
        mentioned_per_record = get_info_per_record(sh_data, record["record_id"], 'cases_mentioned')
        for mention in mentioned_per_record:
            main.add((subject, DATAOUT.mentionedIn, P621[mention]))

    # monitoring activity metadata
    for monitoring in sh_data['monitorings']:
        mon_sub = P621[monitoring['monitoring_id']]
        main.add((mon_sub, RDF.type, DATAOUT.MonitoringActivity))
        main.add((mon_sub, DATAOUT.monitoringCategory, DATAOUT[monitoring['monitoring_cat']]))
        main.add((mon_sub, PROV.wasAssociatedWith, URIRef("https://dataout.org/")))
        main.add((mon_sub, PROV.startedAtTime, Literal(monitoring['started'], datatype=XSD.date)))
        main.add((mon_sub, PROV.endedAtTime, Literal(monitoring['ended'], datatype=XSD.date)))
        main.add((mon_sub, SCHEMA.name, Literal(monitoring['name'], lang="en")))
        main.add((mon_sub, SCHEMA.description, Literal(monitoring['description'], lang="en")))
        for record in sh_data['all_records']:
            if record['monitoring_id'] == monitoring['monitoring_id']:
                main.add((mon_sub, PROV.generated, P621[record['record_id']]))

    # save main KG
    saved = save_kg(main, path_to_save, 'main')

    return saved

### Defendants

In [ ]:
def generate_defendants(sh_data:dict, path_to_save:str) -> str:

    # Namespace declarations
    DATAOUT = Namespace("https://dataout.org/ontology#")
    P621 = Namespace("https://dataout.org/rdf/data/project-6-21#")
    SCHEMA = Namespace("https://schema.org/")
    
    defendants = Graph()

    defendants.bind("dataout", DATAOUT)
    defendants.bind("p621", P621)
    defendants.bind("schema", SCHEMA)
    
    for d in sh_data["defendants"]:
        
        def_sub = P621[str(d["defendant_id"])]
        defendants.add((def_sub, RDF.type, DATAOUT.Defendant))
        defendants.add((def_sub, SCHEMA.identifier, Literal(str(d["defendant_id"]))))
        
        if d["defendant_cat"] == "legal entity":
            defendants.add((def_sub, RDF.type, SCHEMA.Organization))
            defendants.add((def_sub, DATAOUT.defendantCategory, DATAOUT.legalEntity))
            if d["defendant_name_ru"] != '':
                defendants.add((def_sub, SCHEMA.legalName, Literal(d["defendant_name_ru"], lang="ru")))
            if d["defendant_name_en"] != '':
                defendants.add((def_sub, SCHEMA.legalName, Literal(d["defendant_name_en"], lang="en")))
    
        if d["defendant_cat"] == "official":
            defendants.add((def_sub, RDF.type, SCHEMA.Person))
            defendants.add((def_sub, DATAOUT.defendantCategory, DATAOUT.official))
            if d["official_org_id"] != '':
                defendants.add((def_sub, SCHEMA.affiliation, P621[str(d["official_org_id"])]))
            if d["defendant_name_ru"] != '':
                defendants.add((def_sub, SCHEMA.name, Literal(d["defendant_name_ru"], lang="ru")))
            if d["defendant_name_en"] != '':
                defendants.add((def_sub, SCHEMA.name, Literal(d["defendant_name_en"], lang="en")))
    
        if d["defendant_cat"] == "individual":
            defendants.add((def_sub, RDF.type, SCHEMA.Person))
            defendants.add((def_sub, DATAOUT.defendantCategory, DATAOUT.individual))

    # save defendants KG
    saved = save_kg(defendants, path_to_save, 'defendants')
    
    return saved

### Penalties

In [ ]:
def generate_penalties(sh_data:dict, path_to_save:str) -> str:

    # Namespace declarations
    DATAOUT = Namespace("https://dataout.org/ontology#")
    P621 = Namespace("https://dataout.org/rdf/data/project-6-21#")
    SCHEMA = Namespace("https://schema.org/")
    
    penalties = Graph()

    penalties.bind("dataout", DATAOUT)
    penalties.bind("p621", P621)
    penalties.bind("schema", SCHEMA)
    
    for p in sh_data["penalties"]:
        
        pen_sub = P621[p["penalty_id"]]
        penalties.add((pen_sub, RDF.type, DATAOUT.Penalty))
        
        if p["penalty_type"] != "unknown":
            penalties.add((pen_sub, DATAOUT.penaltyCategory, DATAOUT[p["penalty_type"]]))
    
        if p["penalty_type"] == "fine" and p["value"] != "":
            amount = BNode()
            penalties.add((pen_sub, DATAOUT.penaltyValue, amount))
            penalties.add((amount, RDF.type, SCHEMA.MonetaryAmount))
            penalties.add((amount, SCHEMA.value, Literal(p["value"], datatype=XSD.integer)))
            penalties.add((amount, SCHEMA.currency, Literal("RUB")))
    
        if p["penalty_type"] == "detention" and p["value"] != "":
            amount = BNode()
            penalties.add((pen_sub, DATAOUT.penaltyValue, amount))
            penalties.add((amount, RDF.type, SCHEMA.QuantitativeValue))
            penalties.add((amount, SCHEMA.value, Literal(p["value"], datatype=XSD.integer)))
            penalties.add((amount, SCHEMA.unitCode, Literal("DAY")))
            
    # save penalties KG
    saved = save_kg(penalties, path_to_save, 'penalties')
    
    return saved

### Sources

#### also includes Mentioned in

In [ ]:
def generate_sources(sh_data:dict, path_to_save:str) -> str:
    
    category_mappings = {"court":"courtDocument","social media":"socialMediaPost","mass media":"massMediaPublication",
                    "press release":"pressRelease"}

    DATAOUT = Namespace("https://dataout.org/ontology#")
    P621 = Namespace("https://dataout.org/rdf/data/project-6-21#")
    SCHEMA = Namespace("https://schema.org/")
    COURT = Namespace("https://dataout.org/rdf/reference/ru_courts#")
    
    sources = Graph()

    sources.bind("dataout", DATAOUT)
    sources.bind("p621", P621)
    sources.bind("schema", SCHEMA)
    sources.bind("court", COURT)
    
    for s in sh_data["sources"]:
        
        source_sub = P621[s["source_id"]]
        
        sources.add((source_sub, RDF.type, DATAOUT.Source))
        sources.add((source_sub, DATAOUT.sourceCategory, DATAOUT[category_mappings[s["source_category"]]]))
        if s["url"] != "":
            sources.add((source_sub, SCHEMA.url, URIRef(_iri2uri(s["url"]))))
        if s["archived_at"] != "":
            sources.add((source_sub, SCHEMA.archivedAt, URIRef(_iri2uri(s["archived_at"]))))
    
        if s["source_category"] == "court":
            sources.add((source_sub, DCTERMS.creator, COURT[s["court_id"]]))
            sources.add((source_sub, DATAOUT.courtDocumentNumber, Literal(s["court_doc_num"])))
            if s["court_doc_date"] != "":
                sources.add((source_sub, DATAOUT.courtDocumentDate, Literal(s["court_doc_date"], datatype=XSD.date)))
            if s["court_doc_uid"] != "":
                sources.add((source_sub, DATAOUT.uniqueCourtDocumentID, Literal(s["court_doc_uid"])))
    
        else:
            if s["source_name_en"] != "":
                sources.add((source_sub, DCTERMS.creator, Literal(s["source_name_en"], lang="en")))
            if s["source_name_ru"] != "":
                sources.add((source_sub, DCTERMS.creator, Literal(s["source_name_ru"], lang="ru")))
    
    for m in sh_data["mentioned"]:
        
        mentioned_sub = P621[m["mentioned_in_id"]]
    
        sources.add((mentioned_sub, RDF.type, DATAOUT.Source))
        # all mentioned in are mass media
        sources.add((mentioned_sub, DATAOUT.sourceCategory, DATAOUT.massMediaPublication))
        sources.add((mentioned_sub, DCTERMS.language, Literal(m["lang"])))
        if m["source_name_en"] != "":
            sources.add((mentioned_sub, DCTERMS.creator, Literal(m["source_name_en"], lang="en")))
        if m["source_name_ru"] != "":
            sources.add((mentioned_sub, DCTERMS.creator, Literal(m["source_name_ru"], lang="ru")))
        if m["url"] != "":
            sources.add((mentioned_sub, SCHEMA.url, URIRef(_iri2uri(m["url"]))))
        if m["archived_at"] != "":
            sources.add((mentioned_sub, SCHEMA.archivedAt, URIRef(_iri2uri(m["archived_at"]))))

    # save sources KG
    saved = save_kg(sources, path_to_save, 'sources')
    
    return saved

### Trigger objects

In [ ]:
def generate_trigger_objects(sh_data:dict, path_to_save:str) -> str:
    
    triggers_category_mappings = {"Music video":"musicVideo", "Film":"film", "TV series":"tvSeries",\
                              "TV programme":"tvProgramme", "Book":"book", "Comics":"comicBook",\
                              "Protest":"protest", "Public action":"publicAction", "Merchandise":"merchandise",\
                              "Web content":"webContent", "Dating profile":"datingProfile", "Queer venue":"queerVenue",\
                              "Mass media":"massMediaPublication", "Social media":"socialMediaPost"}
    # Namespace declarations
    DATAOUT = Namespace("https://dataout.org/ontology#")
    P621 = Namespace("https://dataout.org/rdf/data/project-6-21#")
    SCHEMA = Namespace("https://schema.org/")
    
    trigger_objects = Graph()
    
    trigger_objects.bind("dataout", DATAOUT)
    trigger_objects.bind("p621", P621)
    trigger_objects.bind("schema", SCHEMA)
    
    for t in sh_data["triggers"]:
        
        trigger_sub = P621[str(t["trigger_object_id"])]
        
        trigger_objects.add((trigger_sub, RDF.type, DATAOUT.TriggerObject))
        trigger_objects.add((trigger_sub, SCHEMA.identifier, Literal(str(t["trigger_object_id"]))))
                             
        if t["trigger_object_cat"] != "Unclear":
            cats_per_object = t["trigger_object_cat"].split(',')
            for cat in cats_per_object:
                trigger_objects.add((trigger_sub, DATAOUT.triggerObjectCategory, DATAOUT[triggers_category_mappings[cat.lstrip(' ')]]))
                
        if t["title_en"] != "":
            trigger_objects.add((trigger_sub, SCHEMA.name, Literal(t["title_en"], lang="en")))
        if t["title_ru"] != "":
            trigger_objects.add((trigger_sub, SCHEMA.name, Literal(t["title_ru"], lang="ru")))
        if t["descr_en"] != "":
            trigger_objects.add((trigger_sub, SCHEMA.description, Literal(t["descr_en"], lang="en")))
        if t["descr_ru"] != "":
            trigger_objects.add((trigger_sub, SCHEMA.description, Literal(t["descr_ru"], lang="ru")))
        if t["year"] != "":
            trigger_objects.add((trigger_sub, SCHEMA.temporal, Literal(t["year"], datatype=XSD.gYear)))
        if t["url"] != "":
            trigger_objects.add((trigger_sub, SCHEMA.url, URIRef(_iri2uri(t["url"]))))

    # save sources KG
    saved = save_kg(trigger_objects, path_to_save, 'trigger_objects')
    
    return saved

## Functions for reference data

### Courts

In [ ]:
def generate_courts(sh_data:dict, path_to_save:str) -> str:

    # Namespace declarations
    DATAOUT = Namespace("https://dataout.org/ontology#")
    SCHEMA = Namespace("https://schema.org/")
    COURT = Namespace("https://dataout.org/rdf/reference/ru_courts#")
    
    courts = Graph()

    courts.bind("dataout", DATAOUT)
    courts.bind("schema", SCHEMA)
    courts.bind("courts", COURT)
    
    for c in sh_data["courts"]:
        
        court_sub = COURT[c["court_id"]]
        courts.add((court_sub, RDF.type, DATAOUT.Court))
        courts.add((court_sub, DATAOUT.courtCode, Literal(c["court_id"])))
        courts.add((court_sub, SCHEMA.name, Literal(c["name_en"], lang="en")))
        courts.add((court_sub, SCHEMA.name, Literal(c["name_ru"], lang="ru")))
        if c["url"] != "":
            courts.add((court_sub, SCHEMA.url, URIRef(_iri2uri(c["url"]))))

    # save courts KG
    saved = save_kg(courts, path_to_save, 'courts')
    
    return saved

### Articles

In [ ]:
def generate_articles(sh_data:dict, path_to_save:str) -> str:
    
    # Namespace declarations
    DATAOUT = Namespace("https://dataout.org/ontology#")
    SCHEMA = Namespace("https://schema.org/")
    LAW = Namespace("https://dataout.org/rdf/reference/ru_law#")
    REGION = Namespace("https://dataout.org/rdf/reference/ru_regions#")
    RDFS = Namespace("http://www.w3.org/2000/01/rdf-schema#")

    articles = Graph()

    articles.bind("dataout", DATAOUT)
    articles.bind("schema", SCHEMA)
    articles.bind("law", LAW)
    articles.bind("region", REGION)

    # add CAORF
    caorf_sub = LAW.CAORF
    articles.add((caorf_sub, RDF.type, SCHEMA.Legislation))
    articles.add((caorf_sub, SCHEMA.legislationType, DATAOUT.administrativeCode))
    articles.add((caorf_sub, SCHEMA.name, Literal("Code of Administrative Offences of the Russian Federation", lang="en")))
    articles.add((caorf_sub, SCHEMA.name, Literal("Кодекс Российской Федерации об административных правонарушениях", lang="ru")))
    articles.add((caorf_sub, SCHEMA.jurisdiction, REGION.RU))
    articles.add((caorf_sub, SCHEMA.hasPart, LAW.article_6_21))
    articles.add((caorf_sub, SCHEMA.hasPart, LAW.article_6_21_2))

    # add articles
    for a in sh_data["articles"]:
        art_sub = LAW[a["article_id"]]
        articles.add((art_sub, RDF.type, SCHEMA.Legislation))
        articles.add((art_sub, SCHEMA.isPartOf, LAW[a["part_of"]]))
        articles.add((art_sub, SCHEMA.name, Literal(a["name_en"], lang="en")))
        articles.add((art_sub, SCHEMA.name, Literal(a["name_ru"], lang="ru")))
        articles.add((art_sub, SCHEMA.description, Literal(a["description_en"], lang="en")))
        articles.add((art_sub, SCHEMA.description, Literal(a["description_ru"], lang="ru")))
        articles.add((art_sub, RDFS.comment, Literal(a["note_en"], lang="en")))
        articles.add((art_sub, RDFS.comment, Literal(a["note_ru"], lang="ru")))

    # save articles KG
    saved = save_kg(articles, path_to_save, 'articles')
    
    return saved

## Exporting files

In [ ]:
file_name = f"-project-6-21-{dt.date.today().isoformat()}"
path_to_save_main = ""
path_to_save_reference = ""
service_account = ""
sh_name = ""

In [ ]:
sh_data = load_spreadsheet(service_account, sh_name)

### Main KG files

In [ ]:
generate_main(sh_data, path_to_save_main)

In [ ]:
generate_defendants(sh_data, path_to_save_main)

In [ ]:
generate_penalties(sh_data, path_to_save_main)

In [ ]:
generate_sources(sh_data, path_to_save_main)

In [ ]:
generate_trigger_objects(sh_data, path_to_save_main)

### Reference data

Note: regions were loaded from TTL and serialised

In [ ]:
regions = Graph()
regions.parse("dataout-org/reference-data/legal-ru/ru_regions.ttl")

In [ ]:
save_kg(regions, path_to_save_reference, 'regions')

In [ ]:
generate_courts(sh_data, path_to_save_reference)

In [ ]:
generate_articles(sh_data, path_to_save_reference)

### Unified KG

In [ ]:
main_ttl_files = [
    "main.ttl",
    "defendants.ttl",
    "penalties.ttl",
    "sources.ttl",
    "trigger_objects.ttl"
]

reference_ttl_files = [
    "regions.ttl",
    "courts.ttl",
    "articles.ttl"
]

In [ ]:
unified_kg = Graph()

In [ ]:
for ttl_file in main_ttl_files:
    file_path = f"{path_to_save_main}/{ttl_file}"
    unified_kg.parse(file_path, format="turtle")
    
for ttt_file_ref in reference_ttl_files:
    file_path = f"{path_to_save_reference}/{ttt_file_ref}"
    unified_kg.parse(file_path, format="turtle")

In [ ]:
# save
save_kg(unified_kg, path_to_save_main, f"kg{file_name}")

### Simplified CSV

In [ ]:
sources_by_id = {source["source_id"]: source for source in sh_data["sources"]}

defendant_categories = {
    defendant["defendant_id"]: defendant["defendant_cat"]
    for defendant in sh_data["defendants"]
}

triggers_by_id = {
    trigger["trigger_object_id"]: trigger
    for trigger in sh_data["triggers"]
}

mentioned_by_id = {
    mention["mentioned_in_id"]: mention
    for mention in sh_data["mentioned"]
}

penalties_by_record = {}
for penalty in sh_data["penalties"]:
    penalties_by_record.setdefault(penalty["record_id"], []).append(penalty)

with open(f"{path_to_save_main}/table{file_name}.csv",'w',encoding="utf-8",newline="") as csv_file:
    
    writer = csv.writer(csv_file)
    header = ["record_id","case_number","year","region","court_code","article","defendant_id","defendant_category",
              "penalty","trigger_object","source","mentioned_in","snippet_en","snippet_ru"]
    writer.writerow(header)
    
    for record in sh_data["all_records"]:
        record_id = record["record_id"]
        sources_per_record = get_info_per_record(sh_data, record_id, "cases_sources")
        court_codes = []
        sources = []
        
        for source_id in sources_per_record:
            source = sources_by_id.get(source_id)
            if source is None:
                continue
            
            if source["source_category"] == "court":
                court_codes.append(source["court_id"])
                if source["court_doc_uid"] != "":
                    single_source_str = f"{source['source_category']} ({source['court_doc_uid']})"
                else:
                    single_source_str = source["source_category"]
            elif source["archived_at"] != "":
                single_source_str = f"{source['source_category']} ({source['archived_at']})"
            else:
                single_source_str = f"{source['source_category']} ({source['url']})"
            sources.append(single_source_str)
        
        court_codes_str = list_to_str(court_codes)
        sources_str = list_to_str(sources)
        defendant_cat = defendant_categories.get(record["defendant_id"], "")
        
        penalties = []
        for penalty in penalties_by_record.get(record_id, []):
            if penalty["penalty_type"] in ("fine", "detention"):
                pen_str = f"{penalty['penalty_type']} ({penalty['value']})"
            else:
                pen_str = penalty["penalty_type"]
            penalties.append(pen_str)
        penalties_str = list_to_str(penalties)
        
        triggers = []
        triggers_per_record = get_info_per_record(sh_data, record_id, "cases_trigger")
        for trigger_id in triggers_per_record:
            trigger = triggers_by_id.get(trigger_id)
            if trigger is not None:
                triggers.append(f"{trigger['trigger_object_cat']} ({trigger['trigger_object_id']})")
        triggers_str = list_to_str(triggers)
        
        mentioned_in = []
        record_mentioned_in = get_info_per_record(sh_data, record_id, "cases_mentioned")
        for mention_id in record_mentioned_in:
            mention = mentioned_by_id.get(mention_id)
            if mention is not None:
                mentioned_in.append(mention["archived_at"] or mention["url"])
        mentioned_in_str = list_to_str(mentioned_in)
        
        row = [record_id, record["case_number"], record["year"], record["region"], court_codes_str, record["article"],
              record["defendant_id"], defendant_cat, penalties_str, triggers_str, sources_str, mentioned_in_str,
              record["snippet_en"], record["snippet_ru"]]
        writer.writerow(row)